# CI/CD for ML with GitHub Actions
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/11_MLOps_Deployment/github_actions_ci_for_ml.ipynb)

Every push should automatically: install deps -> retrain or load the model -> run tests -> FAIL THE BUILD if quality drops. This is the guardrail that stops silent regressions reaching users.

We write a complete workflow file plus the pytest suite it runs - directly usable in this very repository.

## 1. The test suite (quality gate)

In [ ]:
test_code = """
# tests/test_model.py
import pickle
import numpy as np
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression

def train_reference():
    X, y = load_iris(return_X_y=True)
    return LogisticRegression(max_iter=1000).fit(X, y)

def test_accuracy_above_threshold():
    model = train_reference()
    X, y = load_iris(return_X_y=True)
    acc = model.score(X, y)
    assert acc >= 0.90, f"accuracy fell to {acc:.3f}"

def test_prediction_shape_and_labels():
    model = train_reference()
    out = model.predict([[5.1, 3.5, 1.4, 0.2]])
    assert out.shape == (1,)
    assert out[0] in [0, 1, 2]
"""
import os
os.makedirs("tests", exist_ok=True)
open("tests/test_model.py", "w").write(test_code)
print(test_code[:300])

## 2. The workflow file

In [ ]:
workflow = """
# .github/workflows/ml-ci.yml
name: ml-ci

on:
  push: {branches: [main, develop]}
  pull_request:

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
          cache: pip                        # cache pip downloads

      - run: pip install -r requirements.txt scikit-learn pytest

      - name: Run model tests
        run: pytest tests/ -v --tb=short

      # optional: fail on lint too
      - run: pip install ruff && ruff check .
"""
os.makedirs(".github/workflows", exist_ok=True)
open(".github/workflows/ml-ci.yml", "w").write(workflow)
print(workflow)

## 3. What happens after you push

In [ ]:
lifecycle = """
git add tests/ .github/
git commit -m "ci: model quality gate"
git push origin develop

# GitHub then:
#   1. spins an ubuntu VM
#   2. installs python 3.11 + your pinned deps (cached)
#   3. runs pytest -> green check or red X on the commit
#   4. protects branch rules like 'merge only if CI passes'
"""
print(lifecycle)

## Hard-won debugging tips
| Symptom | Usual cause |
|---|---|
| `function main is undeclared` style build errors | CI building something that is not an application - scope jobs to real entry points |
| works locally, fails in CI | version drift - always pin + use `cache: pip` |
| flaky model tests | random seeds unset; fix seeds in tests |
| slow builds | cache dependencies, split heavy training into scheduled (cron) workflows |

Level-up path: matrix testing (py3.10/3.11/3.12) -> nightly full retrains -> deploy job gated on tests -> model registry promotion.